# Batch encoding

- The output of a tokenizer isn’t a simple Python dictionary; what we get is actually a special `BatchEncoding` object.
- It’s a **subclass of a dictionary** (which is why we were able to index into that result without any problem before), but with additional methods that are mostly used by fast tokenizers.

- Besides their **parallelization** capabilities, the key functionality of fast tokenizers is that they always **keep track of the original span** of texts the final tokens come from — a feature we call __offset mapping__.
    - This in turn unlocks features like
        - mapping each word to the tokens it generated
        - or mapping each character of the original text to the token it’s inside, and vice versa.

Let’s take a look at an example:

In [ ]:
from transformers import AutoTokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
print(f"is bert tokenizer fast: {bert_tokenizer.is_fast}")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"is roberta tokenizer fast: {roberta_tokenizer.is_fast}")

## Word <--> Token <--> Sentence

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(type(encoding))
print("====================")
print(f"is fast: {encoding.is_fast}")
print("====================")
print(f"tokens: {encoding.tokens()}")
print("====================")
print(f"word ids: {encoding.word_ids()}")
print("====================")
print(f"sentence ids: {encoding.token_type_ids}")

- The tokenizer’s special tokens `[CLS]` and `[SEP]` are mapped to `None`
- Each token is mapped to the word it originates from.
    - This is especially useful to determine if a token is at the start of a word 
    - or if two tokens are in the same word.
    - this method works for **any** type of tokenizer as long as it’s a **fast** one.
- We can use the `##` prefix to map tokens to words, but it only works for **BERT-like** tokenizers;

==============
- The notion of what a word is complicated.
- For instance, does “I’ll” (a contraction of “I will”) count as one or two words?
    - It actually depends on the tokenizer and the pre-tokenization operation it applies.
    - Some tokenizers just split on spaces, so they will consider this as one word.
    - Others use punctuation on top of spaces, so will consider it two words.
- We can see the difference between `bert` and `roberta` tokenizer for `"81s"`

In [ ]:
example = "81s"
encoding = bert_tokenizer(example)
print(f"bert tokens: {encoding.tokens()}")
print(f"bert word ids: {encoding.word_ids()}")
print("====================")
encoding = roberta_tokenizer(example)
print(f"roberta tokens: {encoding.tokens()}")
print(f"roberta word ids: {encoding.word_ids()}")

## Token <--> Char <--> Word

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(f"words: {encoding.word_ids()}")
print("=====================")
print(f"tokens: {encoding.tokens()}")
print("=====================")
word_id = 3
s, e = encoding.word_to_chars(word_id)
print(f"word number {word_id}: {example[s:e]}")
print("=====================")
token_id = 5
s,e = encoding.token_to_chars(token_id)
print(f"token number {token_id}: {example[s:e]}")
print("=====================")
char_id = 30
print(f"char {char_id} belongs to word {encoding.char_to_word(char_id)} and token {encoding.char_to_token(char_id)}")